# Exercise 6.2: Top-pair production with Pythia

From *Programming in High Energy Particle Physics*, Chapter 6

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch06/ex06_2_solution.ipynb)

Exercise 6.2 Top-pair production with Pythia Use Pythia to generate \(t\bar{t}\) events at \(\sqrt{s} = 13\) TeV. Plot the top quark \(p_T\) distribution, the \(t\bar{t}\) invariant mass, and the jet multiplicity. Compare with theoretical expectations.

<details><summary>Hint</summary>

Start from Listing 6.5 with Top:gg2ttbar = on and Top:qqbar2ttbar = on in place of the Z settings. Find the tops as the last copies of particles with abs(id()) == 6 ( event[i].iBotCopyId() helps) before they decay. For jets, cluster the final-state particles with FastJet (anti-\(k_T\), \(R = 0.4\), see Chapter 8 ) or with Pythia's built-in SlowJet . The \(m_{t\bar{t}}\) distribution starts at \(2m_t \approx 345\) GeV.

</details>

### Colab setup

The first run takes about 3–5 minutes and restarts the runtime once. After restart run all cells from the top. Pythia 8 is installed from conda-forge; this notebook compiles a small C++ program following Listing 6.5, then plots its CSV output in Python.

In [ ]:
import sys, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and importlib.util.find_spec("pythia8") is None:
    %pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
import importlib.util
if IN_COLAB and importlib.util.find_spec("pythia8") is None:
    !mamba install -q -y -c conda-forge pythia8
if IN_COLAB:
    %pip install -q numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt

### Generate top pairs and cluster final-state jets

`SlowJet` counts inclusive jets from final-state particles, including top-decay products. The 25 GeV threshold and acceptance affect the multiplicity.

In [ ]:
%%writefile ex06_2.cc
#include "Pythia8/Pythia.h"
#include <fstream>
#include <iostream>
#include <cmath>
int main() {
  Pythia8::Pythia pythia;
  pythia.readString("Beams:eCM = 13000.");
  pythia.readString("Top:gg2ttbar = on");
  pythia.readString("Top:qqbar2ttbar = on");
  pythia.readString("Random:setSeed = on");
  pythia.readString("Random:seed = 42");
  pythia.readString("Next:numberShowInfo = 0");
  pythia.readString("Next:numberShowProcess = 0");
  pythia.readString("Next:numberShowEvent = 0");
  if(!pythia.init()) return 1;
  Pythia8::SlowJet jetFinder(-1,0.4,25.,2.5); // anti-kT, pT>25 GeV, |eta|<2.5
  std::ofstream out("ex06_2_events.csv");
  out << "top_pt,antitop_pt,tt_mass,n_jets\n";
  const int nEvents=1500; // scale up for smoother spectra
  int accepted=0;
  for(int ev=0;ev<nEvents;++ev) {
    if(!pythia.next()) continue;
    int it=-1,iat=-1;
    for(int i=0;i<pythia.event.size();++i) {
      const auto& p=pythia.event[i];
      if(std::abs(p.id())!=6 || p.iBotCopyId()!=i) continue;
      if(p.id()==6) it=i; else iat=i;
    }
    if(it<0 || iat<0) continue;
    if(!jetFinder.analyze(pythia.event)) continue;
    const auto& top=pythia.event[it];
    const auto& antitop=pythia.event[iat];
    out << top.pT() << ',' << antitop.pT() << ','
        << (top.p()+antitop.p()).mCalc() << ',' << jetFinder.sizeJet() << '\n';
    ++accepted;
  }
  std::cout << "accepted=" << accepted << " / " << nEvents << std::endl;
  pythia.stat();
}


In [ ]:
!c++ -std=c++17 -O2 ex06_2.cc -o ex06_2 $(pythia8-config --cxxflags --libs)
!./ex06_2

In [ ]:
data=np.genfromtxt("ex06_2_events.csv",delimiter=",",names=True)
assert data.size>100
fig,ax=plt.subplots(1,3,figsize=(14,4))
ax[0].hist(np.r_[data["top_pt"],data["antitop_pt"]],bins=35,histtype="step")
ax[0].set(xlabel=r"top $p_T$ [GeV]",ylabel="Top quarks / bin")
ax[1].hist(data["tt_mass"],bins=40,histtype="step")
ax[1].axvline(2*172.5,color="crimson",ls="--",label=r"$2m_t$")
ax[1].set(xlabel=r"$m_{t\bar t}$ [GeV]",ylabel="Events / bin");ax[1].legend()
ax[2].hist(data["n_jets"],bins=np.arange(-0.5,16.5,1),histtype="step")
ax[2].set(xlabel="Jets / event (pT > 25 GeV)",ylabel="Events / bin")
plt.tight_layout();plt.show()
print("minimum m_tt [GeV]:",data["tt_mass"].min())

The spectrum turns on around **2mₜ ≈ 345 GeV**; Pythia’s off-shell top masses can give a few events below that nominal threshold. It falls with increasing mass because of phase space and parton luminosities. Top pₜ is broad; the jet count includes radiation and top-decay products, so it depends strongly on jet threshold and acceptance. These are leading-order Pythia predictions, not a precision cross-section comparison.